In [8]:
import json
import requests
from datetime import datetime, timezone

VILLES = ["paris", "lyon", "marseille", "lille", "toulouse",
          "bordeaux", "nantes", "strasbourg", "nice", "montpellier"]

LAKEHOUSE_ROOT = "/lakehouse/default/Files"

with open(f"{LAKEHOUSE_ROOT}/secrets.json") as f:
    SECRETS = json.load(f)

AQICN_API_KEY = SECRETS["AQICN_API_KEY"]

print(f"Configuration chargée — {len(VILLES)} villes")

StatementMeta(, 7d52bb98-ae59-40d0-b4d5-1993e7cd9ea9, 10, Finished, Available, Finished, False)

Configuration chargée — 10 villes


In [9]:
def extract_aqicn(ville: str) -> dict | None:
    """Appelle l'API AQICN pour une ville. Retourne None si erreur."""
    url = f"https://api.waqi.info/feed/{ville}/?token={AQICN_API_KEY}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
    except requests.RequestException as e:
        print(f"  [KO] {ville} — erreur réseau : {e}")
        return None

    if data.get("status") != "ok":
        print(f"  [KO] {ville} — statut API : {data.get('data')}")
        return None

    return data["data"]

StatementMeta(, 7d52bb98-ae59-40d0-b4d5-1993e7cd9ea9, 11, Finished, Available, Finished, False)

In [11]:
import os

def save_to_bronze(data: dict, source: str, ville: str, ts: datetime) -> str:
    """Écrit le JSON brut dans Files/bronze/{source}/{aaaa}/{mm}/{jj}/{ville}_{hh}h.json"""
    dossier = (f"{LAKEHOUSE_ROOT}/bronze/{source}/"
               f"{ts.year}/{ts.month:02d}/{ts.day:02d}")
    os.makedirs(dossier, exist_ok=True)

    chemin = f"{dossier}/{ville}_{ts.hour:02d}h.json"
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return chemin

StatementMeta(, 7d52bb98-ae59-40d0-b4d5-1993e7cd9ea9, 13, Finished, Available, Finished, False)

In [12]:
timestamp = datetime.now(timezone.utc)
ok, ko = 0, 0

print("=" * 45)
print(f"INGESTION AQICN — {timestamp:%Y-%m-%d %H:%M} UTC")
print("=" * 45)

for ville in VILLES:
    print(f"Traitement : {ville}")
    data = extract_aqicn(ville)
    if data is None:
        ko += 1
        continue
    chemin = save_to_bronze(data, "aqicn", ville, timestamp)
    print(f"  [OK] {chemin.replace(LAKEHOUSE_ROOT, 'Files')}")
    ok += 1

print("-" * 45)
print(f"INGESTION AQICN TERMINÉE — OK : {ok} | KO : {ko}")

StatementMeta(, 7d52bb98-ae59-40d0-b4d5-1993e7cd9ea9, 14, Finished, Available, Finished, False)

INGESTION AQICN — 2026-09-02 10:32 UTC
Traitement : paris
  [OK] Files/bronze/aqicn/2026/09/02/paris_10h.json
Traitement : lyon
  [OK] Files/bronze/aqicn/2026/09/02/lyon_10h.json
Traitement : marseille
  [OK] Files/bronze/aqicn/2026/09/02/marseille_10h.json
Traitement : lille
  [OK] Files/bronze/aqicn/2026/09/02/lille_10h.json
Traitement : toulouse
  [OK] Files/bronze/aqicn/2026/09/02/toulouse_10h.json
Traitement : bordeaux
  [OK] Files/bronze/aqicn/2026/09/02/bordeaux_10h.json
Traitement : nantes
  [OK] Files/bronze/aqicn/2026/09/02/nantes_10h.json
Traitement : strasbourg
  [OK] Files/bronze/aqicn/2026/09/02/strasbourg_10h.json
Traitement : nice
  [OK] Files/bronze/aqicn/2026/09/02/nice_10h.json
Traitement : montpellier
  [OK] Files/bronze/aqicn/2026/09/02/montpellier_10h.json
---------------------------------------------
INGESTION AQICN TERMINÉE — OK : 10 | KO : 0


In [4]:
with open(f"{LAKEHOUSE_ROOT}/secrets.json") as f:
    SECRETS = json.load(f)

StatementMeta(, 7d52bb98-ae59-40d0-b4d5-1993e7cd9ea9, 6, Finished, Available, Finished, False)